# Facial Expression Classification Using LLaVA (Zero-Shot)

This notebook demonstrates **zero-shot facial expression classification** using the [LLaVA v1.6 (Mistral-7B)](https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf) multimodal model.

**Task:** Classify facial images into one of 7 emotion categories:
`angry` | `disgust` | `fear` | `happy` | `neutral` | `sad` | `surprised`

**Approach:** Zero-shot — no fine-tuning or additional training. The model is prompted directly with a classification instruction.

## 1. Imports & Model Loading

We load the **LLaVA-Next** processor and model from Hugging Face. The processor handles image and text preprocessing, while the model generates text outputs based on visual and textual inputs.

- **Precision:** `float16` to reduce GPU memory usage
- **Device:** Automatically selects GPU (CUDA) if available, otherwise CPU

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

# --- Device Selection ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Load Processor & Model ---
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")

model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-mistral-7b-hf",
    torch_dtype=torch.float16,       # Use 16-bit precision to reduce memory usage
    low_cpu_mem_usage=True,           # Minimize CPU RAM consumption during loading
    device_map="auto"                 # Automatically distribute model across available devices
)

Kullanılan cihaz: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

## 2. Zero-Shot Classification

For each uploaded image, we:
1. Convert it to RGB format
2. Construct a prompt asking the model to classify the facial expression
3. Generate a prediction using the LLaVA model
4. Collect all results and export to an Excel file

**Prompt template:**
```
Classify this image into one of the given classes and describe it in one word:
angry, disgust, fear, happy, neutral, sad, surprised
```

In [ ]:
import os
import glob
from PIL import Image
import pandas as pd

# --- Configuration ---
IMAGE_DIR = "../data/test_images/"    # Path to the folder containing input images
OUTPUT_PATH = "../results/classification_results.xlsx"

# Supported image extensions
IMAGE_EXTENSIONS = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]

# --- Collect image paths ---
image_paths = []
for ext in IMAGE_EXTENSIONS:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, ext)))

print(f"Found {len(image_paths)} images in '{IMAGE_DIR}'")

# --- Define prompt ---
prompt = "Classify this image into one of the given classes and describe it in one word: angry, disgust, fear, happy, neutral, sad, surprised"
formatted_prompt = f"[INST] <image>\n{prompt} [/INST]"

# --- Run classification ---
results = []

for image_path in tqdm(image_paths, desc="Classifying"):
    try:
        image_name = os.path.basename(image_path)

        # Open image and convert to RGB
        image = Image.open(image_path).convert("RGB")

        # Preprocess: convert image + text into model-ready tensors
        inputs = processor(images=image, text=formatted_prompt, return_tensors="pt").to(device)

        # Generate prediction
        output = model.generate(**inputs, max_new_tokens=100)
        output_text = processor.decode(output[0], skip_special_tokens=True)

        print(f"{image_name} -> {output_text}")

        results.append({
            "Image Name": image_name,
            "Classification": output_text
        })

    except Exception as e:
        print(f"Skipping {image_name}, error: {e}")
        continue

# --- Export results ---
df = pd.DataFrame(results)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_excel(OUTPUT_PATH, index=False)

print(f"\nAll images processed. Results saved to '{OUTPUT_PATH}'")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saving PrivateTest_104142.jpg to PrivateTest_104142.jpg
Saving PrivateTest_139065.jpg to PrivateTest_139065.jpg
Saving PrivateTest_642696.jpg to PrivateTest_642696.jpg
Saving PrivateTest_914251.jpg to PrivateTest_914251.jpg
Saving PrivateTest_1076980.jpg to PrivateTest_1076980.jpg
Saving PrivateTest_1338609.jpg to PrivateTest_1338609.jpg
Saving PrivateTest_1388103.jpg to PrivateTest_1388103.jpg
Saving PrivateTest_1623206.jpg to PrivateTest_1623206.jpg
Saving PrivateTest_1667417.jpg to PrivateTest_1667417.jpg
Saving PrivateTest_1673508.jpg to PrivateTest_1673508.jpg
Saving PrivateTest_2017514.jpg to PrivateTest_2017514.jpg
Saving PrivateTest_2034433.jpg to PrivateTest_2034433.jpg
Saving PrivateTest_2264189.jpg to PrivateTest_2264189.jpg
Saving PrivateTest_2283840.jpg to PrivateTest_2283840.jpg
Saving PrivateTest_2349643.jpg to PrivateTest_2349643.jpg
Saving PrivateTest_3036992.jpg to PrivateTest_3036992.jpg
Saving PrivateTest_3185131.jpg to PrivateTest_3185131.jpg
Saving PrivateTest_353

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


PrivateTest_104142.jpg -> [INST]  
Classify this image into one of the given classes and describe it in one word: angry, disgust, fear, happy, neutral, sad, surprised [/INST] Surprised 


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


PublicTest_99242645.jpg -> [INST]  
Classify this image into one of the given classes and describe it in one word: angry, disgust, fear, happy, neutral, sad, surprised [/INST] Surprised 
PublicTest_99446963.jpg -> [INST]  
Classify this image into one of the given classes and describe it in one word: angry, disgust, fear, happy, neutral, sad, surprised [/INST] Surprised 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Bütün görüntüler işlendi ve 'classification_results.xlsx' dosyasına kaydedildi.


## 3. Results

The classification results are saved to `results/classification_results.xlsx`.

Each row contains:
- **Image Name** — the filename of the input image
- **Classification** — the model's predicted emotion label

### Limitations
- Zero-shot performance depends heavily on the prompt design
- The model was not fine-tuned on facial expression datasets
- Results may vary with image quality, lighting, and face orientation